# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deeepanshchandra/flyrank-aiml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

### Lane 2: Refresh / Content Opportunity Scoring

**ML task type: Ranking**

The task is to rank pages by how strongly they should be prioritized for human review. Ranking fits this lane because the goal is not simply to assign every page a yes/no label. The useful output is an ordered list so that a reviewer can inspect the highest-priority pages first.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

### Target / proxy

The target would be the starter dataset's `is_declining_label`, which marks whether a page is currently classified as declining based on the defined trend rule. This is a proxy for review priority rather than a direct measurement of whether a future refresh will succeed. The ranking would use this observed label during model development while the final output would remain a prioritization aid for human review.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

### Success metric

The main success metric will be **Precision@50**. It measures how many of the top 50 ranked pages are actually in the target group under the evaluation setup. A higher Precision@50 means the limited review capacity is more concentrated on pages that match the target. The starter Random Forest achieved 0.740 Precision@50 compared with 0.240 for the baseline, so 0.740 is a useful reference point for the next stage, while future evaluation should use a held-out validation setup.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

### Unit of analysis

One row represents **one page/content item** in the starter dataset. The ranking decision is therefore made at the page level: each page receives a set of observable features and can be assigned a priority for review.

In [7]:
from pathlib import Path
import subprocess
import pandas as pd

repo = Path("/content/flyrank-aiml-internship")

if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/deeepanshchandra/flyrank-aiml-internship.git",
            str(repo),
        ],
        check=True,
    )

df = pd.read_csv(repo / "data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.head())

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("\nTarget column preview:")
display(df[["content_id", "trend_direction", "is_declining_label"]].head(10))

Rows: 30,000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Target column preview:


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 5. Why ML beats a fixed rule here

### Why ML instead of a fixed rule?

A fixed rule could rank pages using one signal, such as traffic decline, but page priority depends on several signals at once. Search visibility, traffic, engagement, freshness, content size, and other page-level features can interact in ways that are difficult to capture with a small set of hand-written thresholds. ML can learn combinations of these signals from historical examples and produce a ranking that can be evaluated against held-out data. The output should still support human review rather than automatically deciding what to change.

In [8]:
feature_cols = [
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "word_count",
]

available_features = [c for c in feature_cols if c in df.columns]

print("Signals available for the ranking task:")
print(available_features)

display(df[available_features + ["is_declining_label"]].head())

Signals available for the ranking task:
['search_volume', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'content_age_days', 'word_count']


,search_volume,ctr,avg_position,engagement_rate,scroll_rate,content_age_days,word_count,is_declining_label
0,10.0,0.76,10.6,5.88,4.55,187,3221.0,1
1,90.0,0.05,20.3,0.00,10.00,445,2481.0,1
2,0.0,0.09,36.5,0.00,28.57,141,3515.0,1
3,10.0,0.49,6.2,1.28,3.45,463,NaN,0
4,0.0,0.13,44.0,0.00,24.29,263,2803.0,1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.